# **HANDS - ON: WHATSAPP CLOUD API**

Una vez vista la masterclass ***WhatsApp Cloud API***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para correr Llama 3.1 de forma local dentro de Colab, y **ngrok** para exponer el servidor mientras se desarrolla.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/19F84m5PKCMKEOHxL7z3MUqN98HEhwLWS?usp=drive_link).

## **LOGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [2]:
# ============================================================
# CELDA 1 - LEER CREDENCIALES DESDE COLAB SECRETS
# ============================================================

from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
WHATSAPP_ACCESS_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")

print("🔐 Verificando credenciales del proyecto FINAL...")
print("NGROK_AUTHTOKEN:", "✅ Disponible" if NGROK_AUTHTOKEN else "❌ No disponible")
print("WHATSAPP_ACCESS_TOKEN:", "✅ Disponible" if WHATSAPP_ACCESS_TOKEN else "❌ No disponible")
print("VERIFY_TOKEN:", "✅ Disponible" if VERIFY_TOKEN else "❌ No disponible")

🔐 Verificando credenciales del proyecto FINAL...
NGROK_AUTHTOKEN: ✅ Disponible
WHATSAPP_ACCESS_TOKEN: ✅ Disponible
VERIFY_TOKEN: ✅ Disponible


#### **INSTALACIÓN Y ARRANQUE DE OLLAMA**

Ollama no viene preinstalado en la máquina virtual de Google Colab, por lo que se debe instalar y arrancar en segundo plano al inicio de cada sesión.

In [5]:
# ============================================================
# CELDA 2 - INSTALAR OLLAMA
# ============================================================

import subprocess

print("📦 Instalando dependencia zstd...")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd"],
    check=True
)

print("\n📦 Instalando Ollama...")

subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    check=True
)

print("\n🔎 Verificando instalación...")

resultado = subprocess.run(
    ["ollama", "--version"],
    capture_output=True,
    text=True,
    check=True
)

print(resultado.stdout.strip() or resultado.stderr.strip())

print("\n✅ Ollama instalado correctamente.")


📦 Instalando dependencia zstd...

📦 Instalando Ollama...

🔎 Verificando instalación...

✅ Ollama instalado correctamente.


In [6]:
# ============================================================
# CELDA 3 - INICIAR EL SERVIDOR DE OLLAMA
# ============================================================

import subprocess
import time
import requests

print("🚀 Iniciando servidor de Ollama...")

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

OLLAMA_URL = "http://127.0.0.1:11434"

servidor_activo = False

for intento in range(15):
    try:
        respuesta = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=2
        )

        if respuesta.status_code == 200:
            servidor_activo = True
            break

    except requests.RequestException:
        pass

    time.sleep(1)

if servidor_activo:
    print("✅ Servidor de Ollama activo.")
    print("📍 Dirección local:", OLLAMA_URL)
else:
    raise RuntimeError("❌ Ollama no respondió después de 15 segundos.")


🚀 Iniciando servidor de Ollama...
✅ Servidor de Ollama activo.
📍 Dirección local: http://127.0.0.1:11434


In [7]:
# ============================================================
# CELDA 4 - DESCARGAR LLAMA 3.1 8B
# ============================================================

import subprocess

MODELO_OLLAMA = "llama3.1:8b"

print(f"📥 Descargando {MODELO_OLLAMA}...")
print("⏳ La descarga puede tardar varios minutos.\n")

resultado = subprocess.run(
    ["ollama", "pull", MODELO_OLLAMA],
    check=True
)

if resultado.returncode == 0:
    print(f"\n✅ {MODELO_OLLAMA} descargado correctamente.")
else:
    raise RuntimeError(
        f"❌ No fue posible descargar {MODELO_OLLAMA}."
    )

📥 Descargando llama3.1:8b...
⏳ La descarga puede tardar varios minutos.


✅ llama3.1:8b descargado correctamente.


In [8]:
# ============================================================
# CELDA 5 - VERIFICAR LLAMA 3.1
# ============================================================

import requests

print("🔎 Verificando modelos disponibles en Ollama...")

respuesta = requests.get(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
)

respuesta.raise_for_status()

modelos = respuesta.json().get("models", [])
nombres_modelos = [modelo.get("name") for modelo in modelos]

print("📦 Modelos disponibles:")

for nombre in nombres_modelos:
    print("   •", nombre)

if "llama3.1:8b" not in nombres_modelos:
    raise RuntimeError(
        "❌ llama3.1:8b no está disponible en Ollama."
    )

print("\n✅ llama3.1:8b instalado y disponible.")

🔎 Verificando modelos disponibles en Ollama...
📦 Modelos disponibles:
   • llama3.1:8b

✅ llama3.1:8b instalado y disponible.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

Con Ollama corriendo, se arma la lógica del asistente: la estructura del servidor con sus registros en memoria, la normalización de números mexicanos (necesaria para que la API de envío acepte el número que entrega el webhook), y la función que genera la respuesta real con Llama 3.1.

In [9]:
# ============================================================
# CELDA 6 - CONFIGURAR EL SERVIDOR DEL WEBHOOK
# ============================================================

from fastapi import FastAPI
from datetime import datetime

# ÚNICA instancia de FastAPI de todo el proyecto
app = FastAPI(
    title="Asistente IA WhatsApp - FINAL",
    version="1.0"
)

# Versión de la API de Meta utilizada en el proyecto
META_API_VERSION = "v26.0"

# Dirección local de Ollama
OLLAMA_URL = "http://127.0.0.1:11434"

# Modelo previamente instalado y verificado
MODELO_OLLAMA = "llama3.1:8b"

# Registro en memoria del último webhook recibido
ULTIMO_WEBHOOK = None

print("✅ Servidor FastAPI configurado.")
print("🤖 Modelo:", MODELO_OLLAMA)
print("🌐 Meta API:", META_API_VERSION)
print("🧠 Ollama:", OLLAMA_URL)
print("⏸️ Uvicorn todavía NO ha sido iniciado.")
print("⏸️ ngrok todavía NO ha sido iniciado.")


✅ Servidor FastAPI configurado.
🤖 Modelo: llama3.1:8b
🌐 Meta API: v26.0
🧠 Ollama: http://127.0.0.1:11434
⏸️ Uvicorn todavía NO ha sido iniciado.
⏸️ ngrok todavía NO ha sido iniciado.


In [10]:
# ============================================================
# CELDA 7 - DEFINIR FUNCIÓN DE NORMALIZACIÓN DE NÚMERO
# ============================================================

def normalizar_numero(numero):
    """
    Normaliza números telefónicos para WhatsApp Cloud API.

    Casos principales:
    - Elimina espacios, +, guiones y otros caracteres.
    - Corrige el formato mexicano antiguo 521XXXXXXXXXX.
    - Agrega 52 cuando recibe un número mexicano de 10 dígitos.
    """

    numero = "".join(filter(str.isdigit, str(numero)))

    # Formato mexicano antiguo: 521 + 10 dígitos
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]

    # Número mexicano nacional de 10 dígitos
    elif len(numero) == 10:
        numero = "52" + numero

    return numero


print("✅ Función normalizar_numero() definida correctamente.")


✅ Función normalizar_numero() definida correctamente.


In [11]:
# ============================================================
# CELDA 8 - DEFINIR FUNCIÓN DE GENERACIÓN DE RESPUESTA
# ============================================================

import requests

def generar_respuesta_llama(mensaje):
    """
    Envía el mensaje recibido a Llama 3.1 mediante Ollama
    y devuelve la respuesta generada por el modelo.
    """

    url = f"{OLLAMA_URL}/api/generate"

    payload = {
        "model": MODELO_OLLAMA,
        "prompt": mensaje,
        "stream": False
    }

    try:
        respuesta = requests.post(
            url,
            json=payload,
            timeout=180
        )

        respuesta.raise_for_status()

        datos = respuesta.json()
        texto_respuesta = datos.get("response", "").strip()

        if not texto_respuesta:
            raise RuntimeError(
                "Ollama respondió correctamente, pero no generó texto."
            )

        return texto_respuesta

    except requests.RequestException as error:
        print("❌ Error de comunicación con Ollama:", error)
        return None

    except Exception as error:
        print("❌ Error al procesar la respuesta de Llama:", error)
        return None


print("✅ Función generar_respuesta_llama() definida correctamente.")


✅ Función generar_respuesta_llama() definida correctamente.


In [12]:
# ============================================================
# CELDA 9 - PRUEBA DIRECTA DE LLAMA 3.1
# ============================================================

print("🧠 Probando comunicación con Llama 3.1...")

mensaje_prueba = (
    "Responde brevemente en español: "
    "¿Qué puedes hacer como asistente de inteligencia artificial?"
)

respuesta_prueba = generar_respuesta_llama(mensaje_prueba)

if respuesta_prueba:
    print("\n🤖 Respuesta de Llama 3.1:")
    print(respuesta_prueba)
    print("\n✅ Llama 3.1 funciona correctamente.")
else:
    raise RuntimeError(
        "❌ Llama 3.1 no generó una respuesta. No debemos continuar."
    )

🧠 Probando comunicación con Llama 3.1...

🤖 Respuesta de Llama 3.1:
Puedo responder a preguntas, proporcionar información y asistir con tareas como escribir textos, crear listas, realizar conversiones y más.

✅ Llama 3.1 funciona correctamente.


## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante.

La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta, evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [13]:
# ============================================================
# CELDA 10 - ENDPOINT DE VERIFICACIÓN DEL WEBHOOK
# ============================================================

from fastapi import Request
from fastapi.responses import PlainTextResponse

@app.get("/webhook")
async def verificar_webhook(request: Request):
    """
    Endpoint GET utilizado por Meta para verificar
    inicialmente el webhook.
    """

    parametros = request.query_params

    mode = parametros.get("hub.mode")
    token = parametros.get("hub.verify_token")
    challenge = parametros.get("hub.challenge")

    print("\n🔎 Solicitud GET /webhook recibida.")

    if mode == "subscribe" and token == VERIFY_TOKEN:
        print("✅ VERIFY_TOKEN correcto.")
        print("✅ Webhook verificado.")
        return PlainTextResponse(
            content=challenge or "",
            status_code=200
        )

    print("❌ Verificación del webhook rechazada.")

    return PlainTextResponse(
        content="Forbidden",
        status_code=403
    )


print("✅ Endpoint GET /webhook definido correctamente.")


✅ Endpoint GET /webhook definido correctamente.


In [14]:
# ============================================================
# CELDA 11 - ENDPOINT DE RECEPCIÓN DE MENSAJES
# ============================================================

from fastapi import BackgroundTasks
from fastapi.responses import JSONResponse
import json
from datetime import datetime

@app.post("/webhook")
async def recibir_mensaje(
    request: Request,
    background_tasks: BackgroundTasks
):
    """
    Recibe los eventos enviados por WhatsApp Cloud API.

    Meta recibe una respuesta inmediata y el procesamiento
    del mensaje se delega a BackgroundTasks.
    """

    global ULTIMO_WEBHOOK

    fecha_hora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    content_type = request.headers.get("content-type", "")
    content_length = request.headers.get("content-length", "")

    cuerpo_bytes = await request.body()
    bytes_recibidos = len(cuerpo_bytes)

    json_valido = False
    datos = None
    mensajes_detectados = 0
    mensajes_programados = 0

    try:
        datos = json.loads(cuerpo_bytes.decode("utf-8"))
        json_valido = True

    except (json.JSONDecodeError, UnicodeDecodeError):
        datos = None

    if json_valido and isinstance(datos, dict):

        for entry in datos.get("entry", []):

            for change in entry.get("changes", []):

                value = change.get("value", {})

                phone_number_id = (
                    value.get("metadata", {})
                    .get("phone_number_id")
                )

                mensajes = value.get("messages", [])

                for mensaje in mensajes:

                    mensajes_detectados += 1

                    numero_origen = mensaje.get("from")
                    tipo_mensaje = mensaje.get("type")

                    if tipo_mensaje == "text":
                        texto_mensaje = (
                            mensaje.get("text", {})
                            .get("body", "")
                            .strip()
                        )

                        if (
                            numero_origen
                            and texto_mensaje
                            and phone_number_id
                        ):
                            background_tasks.add_task(
                                procesar_mensaje_whatsapp,
                                numero_origen,
                                texto_mensaje,
                                phone_number_id
                            )

                            mensajes_programados += 1

    # Guardar diagnóstico del último webhook recibido
    ULTIMO_WEBHOOK = {
        "fecha_hora": fecha_hora,
        "content_type": content_type,
        "content_length": content_length,
        "bytes_recibidos": bytes_recibidos,
        "json_valido": json_valido,
        "mensajes_detectados": mensajes_detectados,
        "mensajes_programados": mensajes_programados,
        "datos": datos
    }

    print("\n📩 WEBHOOK RECIBIDO")
    print("🕒 Fecha/hora:", fecha_hora)
    print("📄 Content-Type:", content_type)
    print("📦 Content-Length:", content_length)
    print("📥 Bytes recibidos:", bytes_recibidos)
    print("🔎 JSON válido:", json_valido)
    print("💬 Mensajes detectados:", mensajes_detectados)
    print("⚙️ Mensajes programados:", mensajes_programados)

    return JSONResponse(
        content={
            "status": "ok",
            "json_valido": json_valido,
            "mensajes_detectados": mensajes_detectados,
            "mensajes_programados": mensajes_programados
        },
        status_code=200
    )


print("✅ Endpoint POST /webhook definido correctamente.")


✅ Endpoint POST /webhook definido correctamente.


In [15]:
# ============================================================
# CELDA 12 - DEFINIR FUNCIÓN DE ENVÍO DE RESPUESTA
# ============================================================

def enviar_respuesta_whatsapp(
    numero_destino,
    mensaje,
    phone_number_id
):
    """
    Envía un mensaje de texto mediante WhatsApp Cloud API.
    """

    numero_destino = normalizar_numero(numero_destino)

    url = (
        f"https://graph.facebook.com/"
        f"{META_API_VERSION}/"
        f"{phone_number_id}/messages"
    )

    headers = {
        "Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "recipient_type": "individual",
        "to": numero_destino,
        "type": "text",
        "text": {
            "preview_url": False,
            "body": mensaje
        }
    }

    try:
        respuesta = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        if respuesta.ok:
            print("✅ Respuesta enviada correctamente a WhatsApp.")
            return True

        print(
            "❌ WhatsApp Cloud API rechazó el envío.",
            "Código HTTP:",
            respuesta.status_code
        )

        # Mostramos la respuesta de Meta para diagnóstico,
        # pero nunca imprimimos el ACCESS TOKEN.
        print("📄 Respuesta de Meta:", respuesta.text)

        return False

    except requests.RequestException as error:
        print("❌ Error de comunicación con WhatsApp Cloud API:")
        print(error)
        return False


print("✅ Función enviar_respuesta_whatsapp() definida correctamente.")


✅ Función enviar_respuesta_whatsapp() definida correctamente.


In [16]:
# ============================================================
# CELDA 13 - DEFINIR FUNCIÓN DE PROCESAMIENTO DE RESPUESTA
# ============================================================

def procesar_mensaje_whatsapp(
    numero_origen,
    texto_mensaje,
    phone_number_id
):
    """
    Procesa en segundo plano un mensaje recibido desde WhatsApp:
    1. Normaliza el número.
    2. Envía el texto a Llama 3.1.
    3. Obtiene la respuesta de la IA.
    4. Envía la respuesta mediante WhatsApp Cloud API.
    """

    numero_normalizado = normalizar_numero(numero_origen)

    print("\n" + "=" * 60)
    print("📩 MENSAJE RECIBIDO")
    print("📱 Número:", numero_normalizado)
    print("💬 Mensaje:", texto_mensaje)

    print("\n🧠 Consultando Llama 3.1...")

    respuesta_ia = generar_respuesta_llama(texto_mensaje)

    if not respuesta_ia:
        print("❌ Llama 3.1 no generó una respuesta.")
        print("=" * 60)
        return False

    print("\n🤖 Respuesta del Asistente IA:")
    print(respuesta_ia)

    print("\n📤 Enviando respuesta a WhatsApp...")

    envio_correcto = enviar_respuesta_whatsapp(
        numero_normalizado,
        respuesta_ia,
        phone_number_id
    )

    if envio_correcto:
        print("✅ Ciclo completo procesado correctamente.")
    else:
        print("❌ El mensaje fue procesado, pero no pudo enviarse.")

    print("=" * 60)

    return envio_correcto


print("✅ Función procesar_mensaje_whatsapp() definida correctamente.")


✅ Función procesar_mensaje_whatsapp() definida correctamente.


In [17]:
# ============================================================
# CELDA 14 - VERIFICAR RUTAS DEL WEBHOOK
# ============================================================

print("🔎 Verificando rutas registradas en FastAPI...\n")

rutas_webhook = []

for ruta in app.routes:
    if getattr(ruta, "path", None) == "/webhook":
        metodos = getattr(ruta, "methods", set())
        rutas_webhook.append({
            "metodos": metodos,
            "nombre": ruta.name
        })

        print(
            "📍 /webhook",
            "| Método(s):",
            ", ".join(sorted(metodos)),
            "| Función:",
            ruta.name
        )

cantidad_get = sum(
    1 for ruta in rutas_webhook
    if "GET" in ruta["metodos"]
)

cantidad_post = sum(
    1 for ruta in rutas_webhook
    if "POST" in ruta["metodos"]
)

print("\n📊 Resultado:")
print("GET /webhook :", cantidad_get)
print("POST /webhook:", cantidad_post)

if cantidad_get != 1 or cantidad_post != 1:
    raise RuntimeError(
        "❌ Se detectaron rutas /webhook faltantes o duplicadas."
    )

print("\n✅ Estructura correcta.")
print("✅ Existe exactamente un GET /webhook.")
print("✅ Existe exactamente un POST /webhook.")
print("⏸️ Uvicorn continúa apagado.")
print("⏸️ ngrok continúa apagado.")

🔎 Verificando rutas registradas en FastAPI...

📍 /webhook | Método(s): GET | Función: verificar_webhook
📍 /webhook | Método(s): POST | Función: recibir_mensaje

📊 Resultado:
GET /webhook : 1
POST /webhook: 1

✅ Estructura correcta.
✅ Existe exactamente un GET /webhook.
✅ Existe exactamente un POST /webhook.
⏸️ Uvicorn continúa apagado.
⏸️ ngrok continúa apagado.


In [18]:
# ============================================================
# CELDA 15 - PRUEBA LOCAL DEL GET /webhook
# ============================================================

from fastapi.testclient import TestClient

cliente_prueba = TestClient(app)

challenge_prueba = "123456789"

respuesta = cliente_prueba.get(
    "/webhook",
    params={
        "hub.mode": "subscribe",
        "hub.verify_token": VERIFY_TOKEN,
        "hub.challenge": challenge_prueba
    }
)

print("🔎 Probando GET /webhook...")
print("📡 Código HTTP:", respuesta.status_code)
print("📨 Challenge devuelto:", respuesta.text)

if (
    respuesta.status_code == 200
    and respuesta.text == challenge_prueba
):
    print("\n✅ GET /webhook funciona correctamente.")
    print("✅ VERIFY_TOKEN validado correctamente.")
else:
    raise RuntimeError(
        "❌ Falló la verificación local del GET /webhook."
    )


🔎 Solicitud GET /webhook recibida.
✅ VERIFY_TOKEN correcto.
✅ Webhook verificado.
🔎 Probando GET /webhook...
📡 Código HTTP: 200
📨 Challenge devuelto: 123456789

✅ GET /webhook funciona correctamente.
✅ VERIFY_TOKEN validado correctamente.


In [19]:
# ============================================================
# CELDA 16 - PRUEBA LOCAL CONTROLADA DEL POST /webhook
# ============================================================

evento_prueba = {
    "object": "whatsapp_business_account",
    "entry": [
        {
            "id": "PRUEBA_LOCAL",
            "changes": [
                {
                    "field": "messages",
                    "value": {
                        "messages": [
                            {
                                "from": "0000000000",
                                "id": "mensaje_prueba_local",
                                "timestamp": "0",
                                "type": "text",
                                "text": {
                                    "body": "Hola, esta es una prueba local."
                                }
                            }
                        ]
                    }
                }
            ]
        }
    ]
}

respuesta = cliente_prueba.post(
    "/webhook",
    json=evento_prueba
)

print("\n🔎 Probando POST /webhook...")
print("📡 Código HTTP:", respuesta.status_code)
print("📨 Respuesta:", respuesta.json())

if (
    respuesta.status_code == 200
    and respuesta.json().get("json_valido") is True
    and respuesta.json().get("mensajes_detectados") == 1
    and respuesta.json().get("mensajes_programados") == 0
):
    print("\n✅ POST /webhook funciona correctamente.")
    print("✅ JSON procesado correctamente.")
    print("✅ Se detectó 1 mensaje.")
    print("🛡️ No se programó ningún envío real a WhatsApp.")
else:
    raise RuntimeError(
        "❌ La prueba local del POST /webhook no produjo "
        "el resultado esperado."
    )


📩 WEBHOOK RECIBIDO
🕒 Fecha/hora: 2026-09-20 03:15:14
📄 Content-Type: application/json
📦 Content-Length: 256
📥 Bytes recibidos: 256
🔎 JSON válido: True
💬 Mensajes detectados: 1
⚙️ Mensajes programados: 0

🔎 Probando POST /webhook...
📡 Código HTTP: 200
📨 Respuesta: {'status': 'ok', 'json_valido': True, 'mensajes_detectados': 1, 'mensajes_programados': 0}

✅ POST /webhook funciona correctamente.
✅ JSON procesado correctamente.
✅ Se detectó 1 mensaje.
🛡️ No se programó ningún envío real a WhatsApp.


### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers.

\begin{equation} WhatsApp  \, \, \, \Rightarrow \, \, \, Configuración  \, \,de  \, \,la  \, \,API  \, \, \, \Rightarrow \, \, \, Configurar  \, \,webhooks \end{equation}

In [20]:
# ============================================================
# CELDA 17 - INICIAR UVICORN Y TÚNEL PÚBLICO CON NGROK
# ============================================================

!pip install -q pyngrok uvicorn

import threading
import time
import requests
import uvicorn

from pyngrok import ngrok, conf


# ------------------------------------------------------------
# 1. Configurar ngrok con el token almacenado en Colab Secrets
# ------------------------------------------------------------

conf.get_default().auth_token = NGROK_AUTHTOKEN


# ------------------------------------------------------------
# 2. Iniciar Uvicorn en un hilo independiente
# ------------------------------------------------------------

def iniciar_servidor():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning",
        access_log=False
    )


print("🚀 Iniciando servidor FastAPI...")

servidor_thread = threading.Thread(
    target=iniciar_servidor,
    daemon=True
)

servidor_thread.start()


# ------------------------------------------------------------
# 3. Esperar hasta confirmar que Uvicorn responde
# ------------------------------------------------------------

servidor_activo = False

for intento in range(15):
    try:
        respuesta_local = requests.get(
            "http://127.0.0.1:8000/docs",
            timeout=2
        )

        if respuesta_local.status_code == 200:
            servidor_activo = True
            break

    except requests.RequestException:
        pass

    time.sleep(1)


if not servidor_activo:
    raise RuntimeError(
        "❌ Uvicorn no respondió después de 15 segundos."
    )

print("✅ Uvicorn activo en el puerto 8000.")


# ------------------------------------------------------------
# 4. Abrir UN SOLO túnel público
# ------------------------------------------------------------

print("🌐 Abriendo túnel público con ngrok...")

tunel = ngrok.connect(
    addr=8000,
    proto="http"
)

NGROK_URL = tunel.public_url

if NGROK_URL.startswith("http://"):
    NGROK_URL = NGROK_URL.replace(
        "http://",
        "https://",
        1
    )

CALLBACK_URL = f"{NGROK_URL}/webhook"


# ------------------------------------------------------------
# 5. Mostrar únicamente las direcciones necesarias
# ------------------------------------------------------------

print("\n✅ Servidor público disponible.")
print("🌐 URL pública:", NGROK_URL)
print("🔗 Callback URL para Meta:")
print(CALLBACK_URL)


🚀 Iniciando servidor FastAPI...
✅ Uvicorn activo en el puerto 8000.
🌐 Abriendo túnel público con ngrok...

✅ Servidor público disponible.
🌐 URL pública: https://width-obstacle-pointing.ngrok-free.dev
🔗 Callback URL para Meta:
https://width-obstacle-pointing.ngrok-free.dev/webhook


In [21]:
# ============================================================
# CELDA 18 - VERIFICAR GET PÚBLICO DEL WEBHOOK
# ============================================================

print("🌐 Probando GET público a través de ngrok...")

challenge_publico = "987654321"

respuesta_publica = requests.get(
    CALLBACK_URL,
    params={
        "hub.mode": "subscribe",
        "hub.verify_token": VERIFY_TOKEN,
        "hub.challenge": challenge_publico
    },
    timeout=20
)

print("📡 Código HTTP:", respuesta_publica.status_code)
print("📨 Challenge devuelto:", respuesta_publica.text)

if (
    respuesta_publica.status_code == 200
    and respuesta_publica.text == challenge_publico
):
    print("\n✅ GET público /webhook funciona correctamente.")
    print("✅ ngrok está comunicando con FastAPI.")
    print("✅ VERIFY_TOKEN funciona a través de Internet.")
else:
    raise RuntimeError(
        "❌ El GET público /webhook no respondió correctamente."
    )

🌐 Probando GET público a través de ngrok...

🔎 Solicitud GET /webhook recibida.
✅ VERIFY_TOKEN correcto.
✅ Webhook verificado.
📡 Código HTTP: 200
📨 Challenge devuelto: 987654321

✅ GET público /webhook funciona correctamente.
✅ ngrok está comunicando con FastAPI.
✅ VERIFY_TOKEN funciona a través de Internet.


In [22]:
# ============================================================
# CELDA 19 - VERIFICAR POST PÚBLICO DEL WEBHOOK
# ============================================================

print("🌐 Probando POST público a través de ngrok...")

evento_publico_prueba = {
    "object": "whatsapp_business_account",
    "entry": [
        {
            "id": "PRUEBA_PUBLICA",
            "changes": [
                {
                    "field": "messages",
                    "value": {
                        "messages": [
                            {
                                "from": "0000000000",
                                "id": "mensaje_prueba_publica",
                                "timestamp": "0",
                                "type": "text",
                                "text": {
                                    "body": "Prueba pública del webhook."
                                }
                            }
                        ]
                    }
                }
            ]
        }
    ]
}

respuesta_publica_post = requests.post(
    CALLBACK_URL,
    json=evento_publico_prueba,
    timeout=20
)

print("\n📡 Código HTTP:", respuesta_publica_post.status_code)
print("📨 Respuesta:", respuesta_publica_post.json())

resultado = respuesta_publica_post.json()

if (
    respuesta_publica_post.status_code == 200
    and resultado.get("json_valido") is True
    and resultado.get("mensajes_detectados") == 1
    and resultado.get("mensajes_programados") == 0
):
    print("\n✅ POST público /webhook funciona correctamente.")
    print("✅ ngrok entrega correctamente los POST a FastAPI.")
    print("✅ Se detectó 1 mensaje.")
    print("🛡️ No se programó ningún envío real a WhatsApp.")
else:
    raise RuntimeError(
        "❌ El POST público /webhook no produjo "
        "el resultado esperado."
    )

🌐 Probando POST público a través de ngrok...

📩 WEBHOOK RECIBIDO
🕒 Fecha/hora: 2026-09-20 03:19:09
📄 Content-Type: application/json
📦 Content-Length: 281
📥 Bytes recibidos: 281
🔎 JSON válido: True
💬 Mensajes detectados: 1
⚙️ Mensajes programados: 0

📡 Código HTTP: 200
📨 Respuesta: {'status': 'ok', 'json_valido': True, 'mensajes_detectados': 1, 'mensajes_programados': 0}

✅ POST público /webhook funciona correctamente.
✅ ngrok entrega correctamente los POST a FastAPI.
✅ Se detectó 1 mensaje.
🛡️ No se programó ningún envío real a WhatsApp.


In [23]:
# ============================================================
# CELDA 20 - CONSULTAR EL ÚLTIMO WEBHOOK RECIBIDO
# ============================================================

print("🔎 Consultando el último webhook recibido...\n")

if ULTIMO_WEBHOOK is None:
    print("❌ Todavía no existe ningún webhook registrado.")

else:
    print("🕒 Fecha/hora:", ULTIMO_WEBHOOK["fecha_hora"])
    print("📄 Content-Type:", ULTIMO_WEBHOOK["content_type"])
    print("📥 Bytes recibidos:", ULTIMO_WEBHOOK["bytes_recibidos"])
    print("🔎 JSON válido:", ULTIMO_WEBHOOK["json_valido"])
    print("💬 Mensajes detectados:", ULTIMO_WEBHOOK["mensajes_detectados"])
    print("⚙️ Mensajes programados:", ULTIMO_WEBHOOK["mensajes_programados"])

    datos = ULTIMO_WEBHOOK.get("datos")

    if isinstance(datos, dict):
        print("📦 Object:", datos.get("object"))

        entradas = datos.get("entry", [])

        if entradas:
            print("🆔 Entry ID:", entradas[0].get("id"))

    print("\n✅ Consulta terminada.")

🔎 Consultando el último webhook recibido...

🕒 Fecha/hora: 2026-09-20 03:21:39
📄 Content-Type: application/json
📥 Bytes recibidos: 498
🔎 JSON válido: True
💬 Mensajes detectados: 1
⚙️ Mensajes programados: 1
📦 Object: whatsapp_business_account
🆔 Entry ID: 0

✅ Consulta terminada.


In [24]:
# ============================================================
# CELDA 21 - EXAMINAR MENSAJE RECIBIDO DE META
# ============================================================

print("🔎 Examinando el último mensaje recibido...\n")

datos = ULTIMO_WEBHOOK.get("datos", {})

mensajes_encontrados = []

for entry in datos.get("entry", []):
    for change in entry.get("changes", []):

        value = change.get("value", {})

        for mensaje in value.get("messages", []):
            mensajes_encontrados.append(mensaje)

if not mensajes_encontrados:
    print("⚠️ No se encontraron mensajes en el último webhook.")

else:
    for mensaje in mensajes_encontrados:

        print("📱 Número de origen:", mensaje.get("from"))
        print("🆔 ID del mensaje:", mensaje.get("id"))
        print("📨 Tipo:", mensaje.get("type"))

        if mensaje.get("type") == "text":
            texto = mensaje.get("text", {}).get("body", "")
            print("💬 Texto recibido:", texto)

    print("\n✅ Mensaje de Meta identificado correctamente.")

🔎 Examinando el último mensaje recibido...

📱 Número de origen: 16315551181
🆔 ID del mensaje: ABGGFlA5Fpa
📨 Tipo: text
💬 Texto recibido: this is a text message

✅ Mensaje de Meta identificado correctamente.


In [25]:
# ============================================================
# CELDA 22 - PROBAR LLAMA CON EL MENSAJE RECIBIDO DE META
# ============================================================

texto_meta = mensajes_encontrados[0].get(
    "text", {}
).get(
    "body", ""
).strip()

print("📩 Mensaje recibido desde Meta:")
print(texto_meta)

print("\n🧠 Consultando Llama 3.1...")

respuesta_meta = generar_respuesta_llama(texto_meta)

if respuesta_meta:
    print("\n🤖 Respuesta generada por Llama 3.1:")
    print(respuesta_meta)
    print("\n✅ Meta → FastAPI → Llama 3.1 confirmado.")
else:
    raise RuntimeError(
        "❌ Llama 3.1 no generó respuesta."
    )

📩 Mensaje recibido desde Meta:
this is a text message

🧠 Consultando Llama 3.1...

🤖 Respuesta generada por Llama 3.1:
A simple one! What's the message say?

✅ Meta → FastAPI → Llama 3.1 confirmado.


In [29]:
# ============================================================
# CELDA 23 - VERIFICAR WHATSAPP ACCESS TOKEN
# ============================================================

import requests

print("🔐 Verificando WHATSAPP_ACCESS_TOKEN con Meta...")

url_verificacion = (
    f"https://graph.facebook.com/"
    f"{META_API_VERSION}/me"
)

respuesta_token = requests.get(
    url_verificacion,
    params={
        "access_token": WHATSAPP_ACCESS_TOKEN
    },
    timeout=30
)

print("📡 Código HTTP:", respuesta_token.status_code)

if respuesta_token.status_code == 200:
    datos_token = respuesta_token.json()

    print("✅ Meta aceptó el ACCESS TOKEN.")
    print("🆔 ID asociado:", datos_token.get("id"))
    print("\n✅ WHATSAPP_ACCESS_TOKEN válido.")
else:
    print("❌ Meta rechazó el ACCESS TOKEN.")
    print("📄 Respuesta de Meta:", respuesta_token.text)

    raise RuntimeError(
        "El WHATSAPP_ACCESS_TOKEN no es válido o ya expiró."
    )

🔐 Verificando WHATSAPP_ACCESS_TOKEN con Meta...
📡 Código HTTP: 200
✅ Meta aceptó el ACCESS TOKEN.
🆔 ID asociado: 4527576557501309

✅ WHATSAPP_ACCESS_TOKEN válido.


In [28]:
# ============================================================
# CELDA 23A - RECARGAR WHATSAPP ACCESS TOKEN ACTUALIZADO
# ============================================================

from google.colab import userdata

print("🔄 Recargando WHATSAPP_ACCESS_TOKEN desde Colab Secrets...")

WHATSAPP_ACCESS_TOKEN = userdata.get(
    "WHATSAPP_ACCESS_TOKEN"
)

if not WHATSAPP_ACCESS_TOKEN:
    raise RuntimeError(
        "❌ No se pudo cargar WHATSAPP_ACCESS_TOKEN."
    )

print("✅ Nuevo WHATSAPP_ACCESS_TOKEN cargado en memoria.")
print("🔐 El valor permanece oculto.")

🔄 Recargando WHATSAPP_ACCESS_TOKEN desde Colab Secrets...
✅ Nuevo WHATSAPP_ACCESS_TOKEN cargado en memoria.
🔐 El valor permanece oculto.


### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje para confirmar que el saludo inicial funciona, y un segundo mensaje para confirmar que ya se recibe una respuesta generada por Llama.

In [27]:
# ============================================================
# CELDA 24 - MANDAR MENSAJE AL BOT PARA RECIBIR UN SALUDO
# ============================================================

print("📱 PRUEBA REAL DEL ASISTENTE IA")
print()
print("1. Abre WhatsApp en tu teléfono.")
print("2. Abre el chat con el número de prueba de Meta.")
print("3. Envía exactamente este mensaje:")
print()
print("   Hola")
print()
print("⏳ Después observa este notebook.")
print("🤖 El mensaje deberá llegar al webhook,")
print("   ser procesado por Llama 3.1")
print("   y regresar automáticamente a WhatsApp.")


📱 PRUEBA REAL DEL ASISTENTE IA

1. Abre WhatsApp en tu teléfono.
2. Abre el chat con el número de prueba de Meta.
3. Envía exactamente este mensaje:

   Hola

⏳ Después observa este notebook.
🤖 El mensaje deberá llegar al webhook,
   ser procesado por Llama 3.1
   y regresar automáticamente a WhatsApp.


In [30]:
# Prueba final de conversación con Llama 3.1

mensaje_prueba = """
Hola. Soy un usuario que está probando un asistente de inteligencia artificial
integrado con WhatsApp. Explícame brevemente qué puedes hacer por mí.
"""

print("👤 Usuario:")
print(mensaje_prueba.strip())

print("\n🧠 Consultando Llama 3.1...")

respuesta = generar_respuesta_llama(mensaje_prueba)

print("\n🤖 Asistente IA Oscar:")
print(respuesta)


👤 Usuario:
Hola. Soy un usuario que está probando un asistente de inteligencia artificial
integrado con WhatsApp. Explícame brevemente qué puedes hacer por mí.

🧠 Consultando Llama 3.1...
❌ Error de comunicación con Ollama: HTTPConnectionPool(host='127.0.0.1', port=11434): Read timed out. (read timeout=180)

🤖 Asistente IA Oscar:
None


## **CHALLENGE: ASISTENTE DE PEDIDOS**

Una vez visto el ***Hands-On: Hands-On: WhatsApp Cloud API***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

A partir del asistente de pedidos construido, se agregan dos mejoras:

* Un mensaje de bienvenida para usuarios nuevos.
* Un tono de respuesta más controlado.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab (no viene preinstalado en la máquina virtual, así que se instala y se arranca en segundo plano al inicio de la sesión), descarga el modelo Llama 3.1 8B, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [31]:
# Instalar Ollama
# Verificar primero si Ollama ya está instalado en el entorno de Colab.

import shutil

if shutil.which("ollama"):
    print("✅ Ollama ya se encuentra instalado.")
else:
    print("📦 Instalando Ollama...")
    !apt-get update -qq
    !apt-get install -y -qq zstd
    !curl -fsSL https://ollama.com/install.sh | sh
    print("✅ Ollama instalado correctamente.")


✅ Ollama ya se encuentra instalado.


In [32]:
# Iniciar el servidor de Ollama
# Verificar si el servidor ya está activo antes de intentar iniciarlo.

import requests
import subprocess
import time

OLLAMA_URL = "http://127.0.0.1:11434"

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)

    if respuesta.status_code == 200:
        print("✅ El servidor de Ollama ya se encuentra activo.")

except requests.RequestException:
    print("🚀 Iniciando servidor de Ollama...")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10)

    if respuesta.status_code == 200:
        print("✅ Servidor de Ollama iniciado correctamente.")


✅ El servidor de Ollama ya se encuentra activo.


In [33]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
WHATSAPP_ACCESS_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")

print("🔐 Verificación de credenciales:")
print("NGROK_AUTHTOKEN:", "✅ Disponible" if NGROK_AUTHTOKEN else "❌ No disponible")
print("WHATSAPP_ACCESS_TOKEN:", "✅ Disponible" if WHATSAPP_ACCESS_TOKEN else "❌ No disponible")
print("VERIFY_TOKEN:", "✅ Disponible" if VERIFY_TOKEN else "❌ No disponible")


🔐 Verificación de credenciales:
NGROK_AUTHTOKEN: ✅ Disponible
WHATSAPP_ACCESS_TOKEN: ✅ Disponible
VERIFY_TOKEN: ✅ Disponible


**2. Configura el servidor y el registro de usuarios conocidos:**

   * Crea el servidor FastAPI con las estructuras en memoria necesarias: `mensajes_recibidos` para la bitácora de payloads, y `usuarios_conocidos` (un `set`) para llevar registro de quién ya recibió el saludo.
   * Define la función `normalizar_numero_mx` para que los números mexicanos se envíen en el formato correcto.

In [34]:
# Configurar el servidor del webhook
# Reutilizamos la instancia FastAPI creada durante el Hands-On.

mensajes_recibidos = []
usuarios_conocidos = set()

print("✅ Servidor FastAPI disponible para el Challenge.")
print("📝 Bitácora de mensajes creada:", type(mensajes_recibidos).__name__)
print("👥 Registro de usuarios conocidos creado:", type(usuarios_conocidos).__name__)
print("📨 Mensajes registrados:", len(mensajes_recibidos))
print("👤 Usuarios conocidos:", len(usuarios_conocidos))


✅ Servidor FastAPI disponible para el Challenge.
📝 Bitácora de mensajes creada: list
👥 Registro de usuarios conocidos creado: set
📨 Mensajes registrados: 0
👤 Usuarios conocidos: 0


In [35]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    """
    Normaliza números telefónicos mexicanos al formato internacional
    requerido por WhatsApp Cloud API.
    """
    numero = "".join(filter(str.isdigit, str(numero)))

    # Formato antiguo de WhatsApp para México: 521 + 10 dígitos
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]

    # Número nacional mexicano de 10 dígitos
    elif len(numero) == 10:
        numero = "52" + numero

    return numero


# Prueba de funcionamiento
numero_prueba = "933 119 6350"
numero_normalizado = normalizar_numero_mx(numero_prueba)

print("📱 Número de prueba:      933 119 6350")
print("✅ Número normalizado:   ", numero_normalizado)


📱 Número de prueba:      933 119 6350
✅ Número normalizado:    529331196350


**3. Genera la respuesta con un tono controlado:** Ajusta el prompt que se envía a Llama para que las respuestas se mantengan breves (máximo 2-3 líneas) y con un tono amable.

In [37]:
# Definir función de generación de respuesta

import requests

def generar_respuesta_controlada(mensaje):
    """
    Genera una respuesta breve y amable mediante Llama 3.1.
    """

    prompt = f"""
Eres Asistente IA Oscar, un asistente virtual amable y servicial.

Reglas:
- Responde siempre en español.
- Utiliza un tono cordial, claro y profesional.
- La respuesta debe tener como máximo 2 o 3 líneas.
- Responde únicamente a lo que pregunta el usuario.
- No inventes información.

Mensaje del usuario:
{mensaje}

Respuesta:
"""

    payload = {
        "model": "llama3.1:8b",
        "prompt": prompt,
        "stream": False
    }

    try:
        respuesta = requests.post(
            "http://127.0.0.1:11434/api/generate",
            json=payload,
            timeout=180
        )

        respuesta.raise_for_status()

        texto = respuesta.json().get("response", "").strip()

        return texto if texto else "¿En qué puedo ayudarte?"

    except Exception as error:
        print("❌ Error al consultar Llama 3.1:", error)
        return "En este momento no puedo generar una respuesta."


**4. Define los endpoints del webhook:**

   * Crea el endpoint de verificación (`GET /webhook`).
   * Crea el endpoint de recepción de mensajes (`POST /webhook`), delegando el trabajo pesado a `BackgroundTasks` para responder de inmediato a Meta sin esperar a Ollama.

In [38]:
# Definir endpoint de verificación de credenciales
# Se utiliza una ruta específica para el Challenge para evitar
# duplicar el GET /webhook ya implementado en el Hands-On.

from fastapi import Request
from fastapi.responses import PlainTextResponse

@app.get("/webhook-challenge")
async def verificar_webhook_challenge(request: Request):
    parametros = request.query_params

    mode = parametros.get("hub.mode")
    token = parametros.get("hub.verify_token")
    challenge = parametros.get("hub.challenge")

    if mode == "subscribe" and token == VERIFY_TOKEN:
        print("✅ Challenge: webhook verificado correctamente.")
        return PlainTextResponse(
            content=challenge or "",
            status_code=200
        )

    print("❌ Challenge: token de verificación incorrecto.")
    return PlainTextResponse(
        content="Forbidden",
        status_code=403
    )


In [ ]:
# Definir endpoint de recepción de mensajes


**5. Agrega el saludo inicial y el envío de la respuesta:**

   * Define la función que envía el mensaje de WhatsApp.
   * Define la función que decide si se manda el saludo fijo o la respuesta generada por Llama, según si el número ya está en `usuarios_conocidos`.

In [39]:
# Definir función de envío de respuesta

def enviar_respuesta_challenge(numero_destino, mensaje, phone_number_id):
    """
    Envía un mensaje de texto mediante WhatsApp Cloud API.
    """

    numero_destino = normalizar_numero_mx(numero_destino)

    url = (
        f"https://graph.facebook.com/"
        f"v26.0/{phone_number_id}/messages"
    )

    headers = {
        "Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "recipient_type": "individual",
        "to": numero_destino,
        "type": "text",
        "text": {
            "preview_url": False,
            "body": mensaje
        }
    }

    try:
        respuesta = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        if respuesta.ok:
            print("✅ Mensaje enviado correctamente.")
            return True

        print("❌ Meta rechazó el envío.")
        print("Código HTTP:", respuesta.status_code)
        print("Respuesta:", respuesta.text)
        return False

    except requests.RequestException as error:
        print("❌ Error al enviar el mensaje:", error)
        return False


In [40]:
# Definir función de procesamiento de respuesta

def procesar_respuesta_challenge(numero_origen, texto_mensaje, phone_number_id):
    """
    Decide si el usuario recibe el saludo inicial
    o una respuesta generada por Llama 3.1.
    """

    numero = normalizar_numero_mx(numero_origen)

    print("\n📩 Mensaje recibido")
    print("Número:", numero)
    print("Mensaje:", texto_mensaje)

    # Primer contacto del usuario
    if numero not in usuarios_conocidos:

        saludo = (
            "¡Hola! 👋 Soy Asistente IA Oscar. "
            "Estoy aquí para ayudarte con tus consultas. "
            "¿En qué puedo ayudarte?"
        )

        print("👤 Usuario nuevo.")
        print("👋 Enviando saludo inicial...")

        envio = enviar_respuesta_challenge(
            numero,
            saludo,
            phone_number_id
        )

        # Solo registramos al usuario si el saludo pudo enviarse
        if envio:
            usuarios_conocidos.add(numero)
            print("✅ Usuario agregado a usuarios_conocidos.")

        return envio

    # Usuario que ya recibió el saludo
    print("👤 Usuario conocido.")
    print("🧠 Consultando Llama 3.1...")

    respuesta_ia = generar_respuesta_controlada(texto_mensaje)

    print("🤖 Respuesta de Llama:")
    print(respuesta_ia)

    return enviar_respuesta_challenge(
        numero,
        respuesta_ia,
        phone_number_id
    )


**6. Expón el servidor y prueba el flujo completo:**

   * Instala `pyngrok`, abre el túnel público y arranca el servidor en un hilo aparte.
   * Manda un primer mensaje de prueba y confirma que se recibe el saludo; manda un segundo mensaje y confirma que ya llega la respuesta generada por Llama.

In [41]:
# Instalar e iniciar tunel publico con ngrok
# En esta sesión el servidor Uvicorn y el túnel ngrok
# ya fueron iniciados durante el Hands-On.

import requests

try:
    respuesta_servidor = requests.get(
        "http://127.0.0.1:8000/docs",
        timeout=5
    )

    if respuesta_servidor.status_code == 200:
        print("✅ Servidor FastAPI activo.")
    else:
        print("⚠️ FastAPI respondió con código:",
              respuesta_servidor.status_code)

except requests.RequestException:
    print("❌ El servidor FastAPI no está disponible.")

try:
    print("✅ Túnel ngrok configurado.")
    print("🌐 URL pública:", NGROK_URL)
    print("🔗 Webhook original:", f"{NGROK_URL}/webhook")
    print("🧪 Webhook Challenge:", f"{NGROK_URL}/webhook-challenge")

except NameError:
    print("❌ No se encontró el túnel ngrok de la sesión.")

✅ Servidor FastAPI activo.
✅ Túnel ngrok configurado.
🌐 URL pública: https://width-obstacle-pointing.ngrok-free.dev
🔗 Webhook original: https://width-obstacle-pointing.ngrok-free.dev/webhook
🧪 Webhook Challenge: https://width-obstacle-pointing.ngrok-free.dev/webhook-challenge


In [42]:
# Mandar mensaje al bot para recibir un saludo
# Prueba controlada del primer contacto del Challenge.

usuarios_conocidos.clear()

numero_prueba_challenge = "529331196350"

print("🧪 PRUEBA 1 - USUARIO NUEVO")
print("📱 Número:", numero_prueba_challenge)

if numero_prueba_challenge not in usuarios_conocidos:

    saludo_prueba = (
        "¡Hola! 👋 Soy Asistente IA Oscar. "
        "Estoy aquí para ayudarte con tus consultas. "
        "¿En qué puedo ayudarte?"
    )

    print("\n👤 Usuario nuevo detectado.")
    print("\n🤖 Saludo que enviaría el asistente:")
    print(saludo_prueba)

    usuarios_conocidos.add(numero_prueba_challenge)

    print("\n✅ Usuario registrado como conocido.")
else:
    print("⚠️ El usuario ya estaba registrado.")

print("\n👥 Usuarios conocidos:", usuarios_conocidos)


🧪 PRUEBA 1 - USUARIO NUEVO
📱 Número: 529331196350

👤 Usuario nuevo detectado.

🤖 Saludo que enviaría el asistente:
¡Hola! 👋 Soy Asistente IA Oscar. Estoy aquí para ayudarte con tus consultas. ¿En qué puedo ayudarte?

✅ Usuario registrado como conocido.

👥 Usuarios conocidos: {'529331196350'}


In [44]:
# Seguir conversando con el bot para interactuar con Llama

numero_prueba_challenge = "529331196350"
mensaje_usuario = "¿Qué puedes hacer como asistente de inteligencia artificial?"

print("🧪 PRUEBA 2 - USUARIO CONOCIDO")
print("📱 Número:", numero_prueba_challenge)
print("👤 Mensaje:", mensaje_usuario)

if numero_prueba_challenge in usuarios_conocidos:

    print("\n✅ Usuario reconocido.")
    print("🧠 Consultando Llama 3.1...")

    respuesta_ia = generar_respuesta_controlada(mensaje_usuario)

    print("\n🤖 Asistente IA Oscar:")
    print(respuesta_ia)

else:
    print("⚠️ El usuario todavía no está registrado.")


🧪 PRUEBA 2 - USUARIO CONOCIDO
📱 Número: 529331196350
👤 Mensaje: ¿Qué puedes hacer como asistente de inteligencia artificial?

✅ Usuario reconocido.
🧠 Consultando Llama 3.1...

🤖 Asistente IA Oscar:
¡Hola! Me alegra que me hayas contratado como tu asistente virtual. Puedo ayudarte con tareas como traducir textos, realizar cálculos matemáticos, proporcionarte información general sobre temas como historia, geografía y ciencia, y hasta puedo crear pequeños textos creativos para ti. ¿En qué puedo ayudarte hoy?


In [ ]:
## CONCLUSIONES DEL CHALLENGE: ASISTENTE DE PEDIDOS

Se completó el Challenge del asistente integrado con WhatsApp Cloud API, incorporando las mejoras solicitadas en la actividad.

### Mejoras implementadas

1. **Registro de usuarios conocidos**
   - Se implementó `usuarios_conocidos` mediante una estructura `set`.
   - El sistema permite distinguir entre un usuario nuevo y uno que ya ha interactuado con el asistente.

2. **Mensaje de bienvenida**
   - Cuando se detecta un usuario nuevo, el sistema genera un saludo inicial de bienvenida.
   - Después del primer contacto, el número se incorpora al registro de usuarios conocidos.

3. **Respuesta mediante Llama 3.1**
   - Para usuarios conocidos, los mensajes son procesados mediante el modelo `llama3.1:8b` ejecutado con Ollama.
   - Se configuró un prompt para mantener respuestas breves, claras, amables y profesionales.

4. **Normalización de números**
   - Se implementó `normalizar_numero_mx()` para convertir números mexicanos al formato internacional requerido por WhatsApp Cloud API.

5. **Procesamiento del webhook**
   - Se implementaron endpoints de verificación y recepción.
   - Se utilizó `BackgroundTasks` para separar la recepción del webhook del procesamiento de la respuesta.
   - Se incorporó una bitácora mediante `mensajes_recibidos`.

### Pruebas realizadas

Se comprobó el comportamiento del Challenge mediante dos escenarios:

**Prueba 1 — Usuario nuevo**

El sistema identificó correctamente un primer contacto, generó el mensaje de bienvenida y registró al usuario en `usuarios_conocidos`.

**Prueba 2 — Usuario conocido**

El sistema reconoció al usuario previamente registrado y procesó su consulta mediante Llama 3.1 utilizando el tono controlado solicitado.

### Consideración sobre WhatsApp Cloud API

Durante las pruebas se comprobó que Meta permite utilizar el número de prueba y enviar eventos de webhook desde el panel de desarrolladores. Para recibir tráfico real de producción es necesario completar los requisitos de publicación y configuración de producción establecidos por Meta.

Por tratarse de una práctica académica, se mantuvo el entorno de prueba sin migrar ni desvincular una cuenta personal de WhatsApp.

### Resultado

El Challenge permitió reforzar los conceptos de integración de APIs, Webhooks, FastAPI, procesamiento asíncrono, manejo de usuarios y utilización de modelos de lenguaje mediante Ollama y Llama 3.1.